In [1]:
# DJIWaypointTool

#Purpose: Generate DJI Fly compatible waypoint missions (KMZ/WPML) for DJI Air 3S,
#with interactive map visualization and repeatable parameterized patterns.

#Environment: Python 3.12.10 (DJI Mission Env)

In [2]:
import sys
print(sys.version)

3.12.10 (v3.12.10:0cc81280367, Apr  8 2025, 08:47:00) [Clang 13.0.0 (clang-1300.0.29.30)]


In [3]:
import folium
import shapely
import pyproj
import lxml
import pandas
import numpy
import ipyleaflet

print("All core packages imported successfully.")

All core packages imported successfully.


In [4]:
# Cell 3 (corrected): Interactive map with search + movable pin

from ipyleaflet import Map, Marker, basemaps, basemap_to_tiles, SearchControl, WidgetControl
import ipywidgets as widgets

center_lat = 36.6000
center_lon = -121.9000

from ipywidgets import Layout

m = Map(
    center=(center_lat, center_lon),
    zoom=12,
    scroll_wheel_zoom=True,
    basemap=basemap_to_tiles(basemaps.OpenStreetMap.Mapnik),
    layout=Layout(width="100%", height="1000px")
)

pin = Marker(location=(center_lat, center_lon), draggable=True)
m.add(pin)

status = widgets.HTML()
m.add_control(WidgetControl(widget=status, position="bottomleft"))

def update_status():
    plat, plon = pin.location
    clat, clon = m.center
    status.value = (
        f"<b>Pin:</b> {plat:.6f}, {plon:.6f}"
        f"<br><b>Map center:</b> {clat:.6f}, {clon:.6f}"
        f"<br><b>Tip:</b> search, click map, or drag pin"
    )

# Click-to-move pin
def handle_map_interaction(**kwargs):
    if kwargs.get("type") == "click":
        latlon = kwargs.get("coordinates")
        if latlon:
            pin.location = tuple(latlon)
            update_status()

m.on_interaction(handle_map_interaction)

# Update on pin drag
pin.observe(lambda change: update_status(), names=["location"])

# Search control
search = SearchControl(
    position="topleft",
    url="https://nominatim.openstreetmap.org/search?format=json&q={s}",
    zoom=16,
    marker=None
)

def handle_search_msg(widget, content, buffers):
    loc = None

    if isinstance(content, dict):
        if "location" in content:
            loc = content["location"]
        elif "result" in content and isinstance(content["result"], dict):
            r = content["result"]
            if "lat" in r and "lon" in r:
                loc = (float(r["lat"]), float(r["lon"]))
            elif "y" in r and "x" in r:
                loc = (float(r["y"]), float(r["x"]))

    if loc:
        m.center = tuple(loc)
        pin.location = tuple(loc)
        update_status()

search.on_msg(handle_search_msg)
m.add_control(search)

update_status()
m

Map(center=[36.6, -121.9], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

In [5]:
# Cell 4: Mission parameters (edit these as needed)

pin_lat, pin_lon = pin.location  # current pin position

mission_name = "Test_Lawnmower"
altitude_m = 70          # constant altitude for now
speed_mps = 6            # constant speed for now

width_m = 250
height_m = 150
spacing_m = 25
bearing_deg = 30

print(f"Mission center: {pin_lat:.6f}, {pin_lon:.6f}")
print(f"Pattern: {width_m}m x {height_m}m, spacing {spacing_m}m, bearing {bearing_deg}°")

Mission center: 36.600000, -121.900000
Pattern: 250m x 150m, spacing 25m, bearing 30°


In [6]:
import math
from pyproj import CRS, Transformer
from shapely.geometry import LineString

# ---- Projection helpers ----

def utm_crs_for_latlon(lat: float, lon: float) -> CRS:
    zone = int(math.floor((lon + 180) / 6) + 1)
    is_northern = lat >= 0
    return CRS.from_dict({
        "proj": "utm",
        "zone": zone,
        "south": not is_northern
    })

def make_transformers(lat: float, lon: float):
    utm = utm_crs_for_latlon(lat, lon)
    wgs84 = CRS.from_epsg(4326)
    to_utm = Transformer.from_crs(wgs84, utm, always_xy=True)
    to_wgs84 = Transformer.from_crs(utm, wgs84, always_xy=True)
    return utm, to_utm, to_wgs84


# ---- Lawnmower generator (meter-accurate) ----

def lawnmower_grid(center_lat, center_lon, width_m=200, height_m=120, spacing_m=20, bearing_deg=0):

    _, to_utm, to_wgs84 = make_transformers(center_lat, center_lon)

    cx, cy = to_utm.transform(center_lon, center_lat)

    half_w = width_m / 2
    half_h = height_m / 2

    lines = []
    y = cy - half_h
    direction = 1

    while y <= cy + half_h + 1e-6:
        if direction == 1:
            p1 = (cx - half_w, y)
            p2 = (cx + half_w, y)
        else:
            p1 = (cx + half_w, y)
            p2 = (cx - half_w, y)

        lines.append(LineString([p1, p2]))
        y += spacing_m
        direction *= -1

    # Rotation about center
    theta = math.radians(bearing_deg)
    cos_t, sin_t = math.cos(theta), math.sin(theta)

    def rotate(pt):
        x, y = pt
        dx, dy = x - cx, y - cy
        rx = cx + dx * cos_t - dy * sin_t
        ry = cy + dx * sin_t + dy * cos_t
        return (rx, ry)

    waypoints_utm = []
    for ln in lines:
        for pt in ln.coords:
            waypoints_utm.append(rotate(pt))

    waypoints_ll = []
    for x, y in waypoints_utm:
        lon, lat = to_wgs84.transform(x, y)
        waypoints_ll.append((lat, lon))

    return waypoints_ll

In [7]:
# Cell 5: Generate waypoints from the selected pin

wps = lawnmower_grid(
    center_lat=pin_lat,
    center_lon=pin_lon,
    width_m=width_m,
    height_m=height_m,
    spacing_m=spacing_m,
    bearing_deg=bearing_deg
)

print("Waypoints:", len(wps))
print("First:", wps[0])
print("Last:", wps[-1])

Waypoints: 14
First: (36.5988585263392, -121.90080722116797)
Last: (36.601141467577236, -121.8991927547491)


In [8]:
# === Trail Import (KML/KMZ → Points + Lines, Auto-Zoom, Collapsible Panel) ===
#
# Function:
#   - Import KML/KMZ
#   - Render Placemark Points + LineStrings
#   - Auto-zoom to fit
#   - Persist trail data on map object
#   - Allow panel to collapse to avoid obscuring map

import io
import zipfile
from lxml import etree

import ipywidgets as widgets
from ipyleaflet import LayerGroup, CircleMarker, Polyline, Popup, WidgetControl

# ---------- Overlay ----------
if "trail_overlay" in globals():
    try:
        m.remove_layer(trail_overlay)
    except Exception:
        pass

trail_overlay = LayerGroup()
m.add_layer(trail_overlay)

# ---------- Core UI Widgets ----------
trail_upload = widgets.FileUpload(
    accept=".kml,.kmz",
    multiple=False,
    description="Upload KML/KMZ"
)

trail_clear_btn = widgets.Button(description="Clear Trails")
trail_status = widgets.HTML(value="<b>Trails:</b> none loaded")

# ---------- Collapsible Behavior ----------
toggle_trail_panel_btn = widgets.ToggleButton(
    value=True,
    description="Hide Trail Panel",
    tooltip="Hide/show trail import controls"
)

trail_controls_box = widgets.VBox(
    [
        widgets.HTML("<b>Trail Import</b>"),
        widgets.HTML("Loads Point markers and LineString trails."),
        trail_upload,
        trail_clear_btn,
        trail_status,
    ],
    layout=widgets.Layout(width="340px")
)

trail_panel_container = widgets.VBox(
    [widgets.HBox([toggle_trail_panel_btn]), trail_controls_box],
    layout=widgets.Layout(width="360px")
)

if "trail_panel_control" in globals():
    try:
        m.remove_control(trail_panel_control)
    except Exception:
        pass

trail_panel_control = WidgetControl(widget=trail_panel_container, position="bottomright")
m.add_control(trail_panel_control)

def _update_trail_panel_visibility(change=None):
    if toggle_trail_panel_btn.value:
        toggle_trail_panel_btn.description = "Hide Trail Panel"
        trail_controls_box.layout.display = "flex"
        trail_panel_container.layout.width = "360px"
    else:
        toggle_trail_panel_btn.description = "Show Trail Panel"
        trail_controls_box.layout.display = "none"
        trail_panel_container.layout.width = "160px"

toggle_trail_panel_btn.observe(_update_trail_panel_visibility, names=["value"])
_update_trail_panel_visibility()

# ---------- Helpers ----------
def _clear_trail_overlay():
    global trail_overlay
    try:
        m.remove_layer(trail_overlay)
    except Exception:
        pass
    trail_overlay = LayerGroup()
    m.add_layer(trail_overlay)

def _extract_kml_bytes(file_name, raw_bytes):
    lower = (file_name or "").lower()
    if lower.endswith(".kmz"):
        with zipfile.ZipFile(io.BytesIO(raw_bytes), "r") as zf:
            kml_files = [n for n in zf.namelist() if n.lower().endswith(".kml")]
            if not kml_files:
                raise ValueError("KMZ contains no .kml file.")
            return zf.read(kml_files[0])
    if lower.endswith(".kml"):
        return raw_bytes
    raise ValueError("Unsupported file type.")

def _ns(root):
    nsmap = getattr(root, "nsmap", {}) or {}
    default_ns = nsmap.get(None, "http://www.opengis.net/kml/2.2")
    return {"k": default_ns}

def _parse_point_coords(text):
    first = text.strip().split()[0]
    parts = first.split(",")
    if len(parts) < 2:
        return None
    lon = float(parts[0])
    lat = float(parts[1])
    return (lat, lon)

def _parse_linestring_coords(text):
    pts = []
    for tup in text.strip().split():
        parts = tup.split(",")
        if len(parts) < 2:
            continue
        lon = float(parts[0])
        lat = float(parts[1])
        pts.append((lat, lon))
    return pts if len(pts) >= 2 else None

def _fit_bounds(latlons):
    lats = [p[0] for p in latlons]
    lons = [p[1] for p in latlons]
    if not lats:
        return
    m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

def _parse_kml(kml_bytes):
    root = etree.fromstring(kml_bytes)
    ns = _ns(root)

    placemarks = root.xpath(".//k:Placemark", namespaces=ns) or root.xpath(".//Placemark")

    points = []
    lines = []

    for pm in placemarks:
        name = "Placemark"
        name_els = pm.xpath("./k:name", namespaces=ns) or pm.xpath("./name")
        if name_els and name_els[0].text:
            name = name_els[0].text.strip()

        # Points
        pcoords = pm.xpath(".//k:Point/k:coordinates", namespaces=ns) or pm.xpath(".//Point/coordinates")
        if pcoords and pcoords[0].text:
            latlon = _parse_point_coords(pcoords[0].text)
            if latlon:
                points.append({"name": name, "lat": latlon[0], "lon": latlon[1]})

        # LineStrings
        lcoords = pm.xpath(".//k:LineString/k:coordinates", namespaces=ns) or pm.xpath(".//LineString/coordinates")
        if lcoords and lcoords[0].text:
            pts = _parse_linestring_coords(lcoords[0].text)
            if pts:
                lines.append({"name": name, "coords": pts})

    return points, lines

def _read_upload_payload(uploader):
    val = uploader.value

    if isinstance(val, dict) and val:
        name = next(iter(val.keys()))
        raw = val[name]["content"]
        return name, raw

    if isinstance(val, (list, tuple)) and len(val) > 0:
        payload = val[0]
        return payload.get("name", "uploaded"), payload.get("content")

    raise ValueError("Empty upload payload.")

def _render(points, lines):
    _clear_trail_overlay()
    all_coords = []

    for ln in lines:
        coords = ln["coords"]
        all_coords.extend(coords)
        trail_overlay.add_layer(Polyline(locations=coords, weight=3))

    for p in points:
        lat, lon = p["lat"], p["lon"]
        all_coords.append((lat, lon))
        marker = CircleMarker(location=(lat, lon), radius=6)
        marker.popup = Popup(
            child=widgets.HTML(value=f"<b>{p['name']}</b><br>{lat:.6f}, {lon:.6f}")
        )
        trail_overlay.add_layer(marker)

    if all_coords:
        _fit_bounds(all_coords)

    m._trail_points = points
    m._trail_lines = lines

    trail_status.value = f"<b>Trails:</b> loaded {len(points)} point(s), {len(lines)} line(s)."

# ---------- Events ----------
def _on_upload_change(change):
    if not trail_upload.value:
        return
    try:
        file_name, raw_bytes = _read_upload_payload(trail_upload)
        kml_bytes = _extract_kml_bytes(file_name, raw_bytes)
        points, lines = _parse_kml(kml_bytes)
        if not points and not lines:
            trail_status.value = "<b>Trails:</b> No supported Placemarks found."
            return
        _render(points, lines)
    except Exception as e:
        trail_status.value = f"<b>Trails:</b> Error: {str(e)}"

def _on_clear_clicked(b=None):
    _clear_trail_overlay()
    trail_status.value = "<b>Trails:</b> cleared."

trail_upload.observe(_on_upload_change, names=["value"])
trail_clear_btn.on_click(_on_clear_clicked)

m

Map(bottom=410092.0, center=[36.6, -121.9], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

In [9]:
# === Integrated UI: Map + Waypoint Planner + Trail Import (KML/KMZ) ===
#
# This cell is self-contained and intended to be the last cell you run.
# It does not depend on earlier cells in the notebook.

import io
import zipfile
import math

import ipywidgets as widgets
from ipywidgets import Layout
from lxml import etree
from pyproj import CRS, Transformer

from ipyleaflet import (
    Map, basemaps, basemap_to_tiles,
    SearchControl, WidgetControl,
    LayerGroup, Polyline, CircleMarker, Popup, Marker
)

from IPython.display import clear_output

# -----------------------------
# Hard reset of prior widget state (prevents "vestigial" controls)
# -----------------------------
clear_output(wait=True)

try:
    # If a previous run left a live map widget, close it to prevent orphan controls
    if "m" in globals() and hasattr(m, "close"):
        m.close()
except Exception:
    pass


# -----------------------------
# Map
# -----------------------------
default_center = (8.614418, -77.857103)
m = Map(
    center=default_center,
    zoom=12,
    scroll_wheel_zoom=True,
    basemap=basemap_to_tiles(basemaps.OpenStreetMap.Mapnik),
    layout=Layout(width="100%", height="650px")
)

m.add_control(SearchControl(
    position="topleft",
    url="https://nominatim.openstreetmap.org/search?format=json&q={s}",
    zoom=14
))


# -----------------------------
# Waypoint Planner (existing behavior preserved, panel default hidden)
# -----------------------------
# CRS + transformer for local meter offsets
crs_wgs84 = CRS.from_epsg(4326)
crs_utm = CRS.from_epsg(32618)  # keep whatever you had, adjust if needed
to_utm = Transformer.from_crs(crs_wgs84, crs_utm, always_xy=True)
to_wgs84 = Transformer.from_crs(crs_utm, crs_wgs84, always_xy=True)

# Layers for flight path display
if "flight_overlay" in globals():
    try:
        m.remove_layer(flight_overlay)
    except Exception:
        pass

flight_overlay = LayerGroup()
m.add_layer(flight_overlay)

launch_marker = None
waypoint_markers = []
path_polyline = None

# UI widgets (you may already have more; keep minimal, functional core here)
wp_status = widgets.HTML(value="<b>Flight Path:</b> no launch point set")
wp_clear_btn = widgets.Button(description="Clear Flight Path", layout=Layout(width="160px"))

toggle_wp_panel_btn = widgets.ToggleButton(
    value=False,
    description="Show Waypoint Panel",
    tooltip="Hide/show waypoint planner controls"
)

wp_controls_box = widgets.VBox(
    [
        widgets.HTML("<b>Waypoint Planner</b>"),
        widgets.HTML("Click map to set launch point. (Stub for your full lawnmower planner UI.)"),
        wp_clear_btn,
        wp_status,
    ],
    layout=widgets.Layout(width="340px")
)

wp_panel_container = widgets.VBox(
    [widgets.HBox([toggle_wp_panel_btn]), wp_controls_box],
    layout=widgets.Layout(width="360px")
)

def _update_wp_panel_visibility(change=None):
    if toggle_wp_panel_btn.value:
        toggle_wp_panel_btn.description = "Hide Waypoint Panel"
        wp_controls_box.layout.display = "flex"
        wp_panel_container.layout.width = "360px"
    else:
        toggle_wp_panel_btn.description = "Show Waypoint Panel"
        wp_controls_box.layout.display = "none"
        wp_panel_container.layout.width = "170px"

toggle_wp_panel_btn.observe(_update_wp_panel_visibility, names=["value"])
_update_wp_panel_visibility()

# Remove any existing waypoint panel controls if rerun without kernel restart
def _remove_panel_controls_by_text(map_obj, text_substring):
    try:
        for c in list(map_obj.controls):
            if isinstance(c, WidgetControl):
                w = getattr(c, "widget", None)
                if w is None:
                    continue

                # Walk one level deep for a ToggleButton with matching description
                if hasattr(w, "children"):
                    try:
                        for ch in w.children:
                            if hasattr(ch, "description") and text_substring in (ch.description or ""):
                                map_obj.remove_control(c)
                                break
                    except Exception:
                        pass
    except Exception:
        pass

_remove_panel_controls_by_text(m, "Waypoint Panel")

wp_panel_control = WidgetControl(widget=wp_panel_container, position="topright")
m.add_control(wp_panel_control)

def _clear_flight_path(b=None):
    global launch_marker, waypoint_markers, path_polyline

    try:
        if launch_marker is not None:
            flight_overlay.remove_layer(launch_marker)
    except Exception:
        pass
    launch_marker = None

    for mk in list(waypoint_markers):
        try:
            flight_overlay.remove_layer(mk)
        except Exception:
            pass
    waypoint_markers = []

    try:
        if path_polyline is not None:
            flight_overlay.remove_layer(path_polyline)
    except Exception:
        pass
    path_polyline = None

    wp_status.value = "<b>Flight Path:</b> cleared"

wp_clear_btn.on_click(_clear_flight_path)

def _on_map_click(**kwargs):
    global launch_marker
    if kwargs.get("type") != "click":
        return
    latlon = kwargs.get("coordinates")
    if not latlon:
        return

    # Clear any prior launch marker
    if launch_marker is not None:
        try:
            flight_overlay.remove_layer(launch_marker)
        except Exception:
            pass

    launch_marker = Marker(location=latlon, draggable=False)
    flight_overlay.add_layer(launch_marker)
    wp_status.value = f"<b>Flight Path:</b> launch point set at {latlon[0]:.6f}, {latlon[1]:.6f}"

m.on_interaction(_on_map_click)


# -----------------------------
# Trail Import (KML/KMZ), multi-layer with per-layer color
# -----------------------------
trail_upload = widgets.FileUpload(accept=".kml,.kmz", multiple=True, description="Upload KML/KMZ")
trail_clear_btn = widgets.Button(description="Clear All Trails", layout=Layout(width="160px"))
trail_status = widgets.HTML(value="<b>Trails:</b> none loaded")

toggle_trail_panel_btn = widgets.ToggleButton(
    value=False,
    description="Show Trail Panel",
    tooltip="Hide/show trail import controls"
)

layers_box = widgets.VBox([], layout=Layout(width="340px"))

trail_controls_box = widgets.VBox(
    [
        widgets.HTML("<b>Trail Import</b>"),
        widgets.HTML("Each upload becomes its own layer, with an independent color."),
        trail_upload,
        trail_clear_btn,
        trail_status,
        widgets.HTML("<b>Loaded Layers</b>"),
        layers_box,
    ],
    layout=widgets.Layout(width="340px")
)

trail_panel_container = widgets.VBox(
    [widgets.HBox([toggle_trail_panel_btn]), trail_controls_box],
    layout=widgets.Layout(width="360px")
)

def _update_trail_panel_visibility(change=None):
    if toggle_trail_panel_btn.value:
        toggle_trail_panel_btn.description = "Hide Trail Panel"
        trail_controls_box.layout.display = "flex"
        trail_panel_container.layout.width = "360px"
    else:
        toggle_trail_panel_btn.description = "Show Trail Panel"
        trail_controls_box.layout.display = "none"
        trail_panel_container.layout.width = "170px"

toggle_trail_panel_btn.observe(_update_trail_panel_visibility, names=["value"])
_update_trail_panel_visibility()

# Layer registry on the map object
if not hasattr(m, "_trail_layers"):
    m._trail_layers = {}  # key -> dict(meta, layergroup, ui_row)

def _extract_kml_bytes(file_name, raw_bytes):
    lower = (file_name or "").lower()
    if lower.endswith(".kmz"):
        with zipfile.ZipFile(io.BytesIO(raw_bytes), "r") as zf:
            kml_files = [n for n in zf.namelist() if n.lower().endswith(".kml")]
            if not kml_files:
                raise ValueError("KMZ contains no .kml file.")
            return zf.read(kml_files[0])
    if lower.endswith(".kml"):
        return raw_bytes
    raise ValueError("Unsupported file type.")

def _ns(root):
    nsmap = getattr(root, "nsmap", {}) or {}
    default_ns = nsmap.get(None, "http://www.opengis.net/kml/2.2")
    return {"k": default_ns}

def _parse_point_coords(text):
    first = text.strip().split()[0]
    parts = first.split(",")
    if len(parts) < 2:
        return None
    lon = float(parts[0])
    lat = float(parts[1])
    return (lat, lon)

def _parse_linestring_coords(text):
    pts = []
    for tup in text.strip().split():
        parts = tup.split(",")
        if len(parts) < 2:
            continue
        lon = float(parts[0])
        lat = float(parts[1])
        pts.append((lat, lon))
    return pts if len(pts) >= 2 else None

def _fit_bounds(latlons):
    lats = [p[0] for p in latlons]
    lons = [p[1] for p in latlons]
    if not lats:
        return
    m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

def _parse_kml(kml_bytes):
    root = etree.fromstring(kml_bytes)
    ns = _ns(root)
    placemarks = root.xpath(".//k:Placemark", namespaces=ns) or root.xpath(".//Placemark")

    points = []
    lines = []

    for pm in placemarks:
        name = "Placemark"
        name_els = pm.xpath("./k:name", namespaces=ns) or pm.xpath("./name")
        if name_els and name_els[0].text:
            name = name_els[0].text.strip()

        pcoords = pm.xpath(".//k:Point/k:coordinates", namespaces=ns) or pm.xpath(".//Point/coordinates")
        if pcoords and pcoords[0].text:
            latlon = _parse_point_coords(pcoords[0].text)
            if latlon:
                points.append({"name": name, "lat": latlon[0], "lon": latlon[1]})

        lcoords = pm.xpath(".//k:LineString/k:coordinates", namespaces=ns) or pm.xpath(".//LineString/coordinates")
        if lcoords and lcoords[0].text:
            pts = _parse_linestring_coords(lcoords[0].text)
            if pts:
                lines.append({"name": name, "coords": pts})

    return points, lines

def _refresh_layers_box():
    layers_box.children = [rec["ui_row"] for rec in m._trail_layers.values()]

def _update_status():
    n = len(m._trail_layers)
    if n == 0:
        trail_status.value = "<b>Trails:</b> none loaded"
    else:
        trail_status.value = f"<b>Trails:</b> {n} layer(s) loaded"

def _apply_color_to_layer(layer_group, color_hex):
    for lyr in getattr(layer_group, "layers", []):
        if isinstance(lyr, Polyline):
            lyr.color = color_hex
        elif isinstance(lyr, CircleMarker):
            lyr.color = color_hex
            lyr.fill_color = color_hex

def _set_layer_visibility(layer_group, visible: bool):
    for lyr in getattr(layer_group, "layers", []):
        lyr.visible = bool(visible)

def _remove_trail_layer(key):
    rec = m._trail_layers.get(key)
    if not rec:
        return
    lg = rec["layer_group"]
    try:
        m.remove_layer(lg)
    except Exception:
        pass
    del m._trail_layers[key]
    _refresh_layers_box()
    _update_status()

def _add_trail_layer(file_name, raw_bytes, default_color="#FF0000"):
    kml_bytes = _extract_kml_bytes(file_name, raw_bytes)
    points, lines = _parse_kml(kml_bytes)
    if not points and not lines:
        raise ValueError("No supported Placemarks found (Points or LineStrings).")

    lg = LayerGroup()
    all_coords = []

    for ln in lines:
        coords = ln["coords"]
        all_coords.extend(coords)
        lg.add_layer(Polyline(locations=coords, weight=3, color=default_color))

    for p in points:
        lat, lon = p["lat"], p["lon"]
        all_coords.append((lat, lon))
        marker = CircleMarker(location=(lat, lon), radius=6, color=default_color, fill_color=default_color)
        marker.popup = Popup(child=widgets.HTML(value=f"<b>{p['name']}</b><br>{lat:.6f}, {lon:.6f}"))
        lg.add_layer(marker)

    m.add_layer(lg)

    label = widgets.HTML(value=f"{file_name}")
    color = widgets.ColorPicker(value=default_color, description="", layout=Layout(width="110px"))
    visible = widgets.Checkbox(value=True, description="Visible", indent=False, layout=Layout(width="90px"))
    remove_btn = widgets.Button(description="Remove", layout=Layout(width="70px"))

    key = f"{file_name}:{len(raw_bytes)}"

    def _on_color(change):
        _apply_color_to_layer(lg, change["new"])

    def _on_visible(change):
        _set_layer_visibility(lg, change["new"])

    def _on_remove(b=None):
        _remove_trail_layer(key)

    color.observe(_on_color, names=["value"])
    visible.observe(_on_visible, names=["value"])
    remove_btn.on_click(_on_remove)

    row = widgets.HBox([label, color, visible, remove_btn], layout=Layout(width="340px"))

    m._trail_layers[key] = {
        "file_name": file_name,
        "n_bytes": len(raw_bytes),
        "layer_group": lg,
        "ui_row": row
    }

    _refresh_layers_box()
    _update_status()

    if all_coords:
        _fit_bounds(all_coords)

def _on_trail_upload_change(change):
    if not trail_upload.value:
        return

    payloads = []
    val = trail_upload.value

    if isinstance(val, dict):
        for name, meta in val.items():
            payloads.append((name, meta.get("content", None)))
    elif isinstance(val, (list, tuple)):
        for item in val:
            payloads.append((item.get("name", "uploaded"), item.get("content", None)))

    added = 0
    errors = []

    for file_name, raw_bytes in payloads:
        if raw_bytes is None:
            errors.append(f"{file_name}: missing content bytes")
            continue

        key = f"{file_name}:{len(raw_bytes)}"
        if key in m._trail_layers:
            continue

        try:
            defaults = ["#FF0000", "#00AA00", "#0066FF", "#FF9900", "#AA00AA", "#00AAAA"]
            default_color = defaults[len(m._trail_layers) % len(defaults)]
            _add_trail_layer(file_name, raw_bytes, default_color=default_color)
            added += 1
        except Exception as e:
            errors.append(f"{file_name}: {str(e)}")

    if added == 0 and not errors:
        trail_status.value = "<b>Trails:</b> no new files added (duplicates skipped)."
    elif errors:
        trail_status.value = "<b>Trails:</b> added some layers, with errors:<br>" + "<br>".join(errors)

def _on_trail_clear_clicked(b=None):
    keys = list(m._trail_layers.keys())
    for k in keys:
        _remove_trail_layer(k)
    _update_status()

trail_upload.observe(_on_trail_upload_change, names=["value"])
trail_clear_btn.on_click(_on_trail_clear_clicked)

# Remove any existing Trail Panel controls that might persist (extra safety)
_remove_panel_controls_by_text(m, "Trail Panel")

trail_panel_control = WidgetControl(widget=trail_panel_container, position="bottomright")
m.add_control(trail_panel_control)

m

Map(center=[8.614418, -77.857103], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title',…